# MMS Stokes — Method of Manufactured Solutions

**Stokes equations on the unit square**: −νΔu + ∇p = f, div u = 0, ν = 1.

**Manufactured solution**:
- Stream function ψ = sin²(πx)sin²(πy) + x²y → div u = 0 by construction
- u_x = ∂ψ/∂y = π·sin²(πx)·sin(2πy) + x²
- u_y = −∂ψ/∂x = −π·sin(2πx)·sin²(πy) − 2xy
- p = cos(πx)sin(πy)  (mean = 0 over [0,1]²)

**Scheme**: Taylor-Hood P2/P1, residual form + Newton, structured triangular meshes N×N,
N = 8, 16, 32, 64, 128.

**Reference rates**: ‖u‖_L2 → 3, ‖u‖_H1 → 2, ‖p‖_L2 → 2 (optimal for TH P2/P1).

Two BC methods are compared:
- **penalty**: γ(u−g, v)_∂Ω — non-symmetric, consistency error O(h/C)
- **nitsche**: symmetric Nitsche with traction terms — optimal for all C above threshold

Errors measured against exact solution via UFL SpatialCoordinate expressions;
f and g enter the forms directly as UFL (no nodal interpolation);
quadrature degree = 2k+3 = 7 (k = 2 for P2 velocity).

In [ ]:
# --- canonical: bootstrap v1 ---
import sys, os, subprocess

_on_colab = "google.colab" in sys.modules

if not _on_colab:
    os.environ.setdefault("OMP_NUM_THREADS", "1")

if _on_colab:
    try:
        import gmsh
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/gmsh-install.sh"'
            ' -O /tmp/gmsh-install.sh && bash /tmp/gmsh-install.sh',
            shell=True, check=True,
        )
    try:
        import dolfinx
    except ImportError:
        subprocess.run(
            'wget -q "https://fem-on-colab.github.io/releases/fenicsx-install-release-real.sh"'
            ' -O /tmp/fenicsx-install.sh && bash /tmp/fenicsx-install.sh',
            shell=True, check=True,
        )

import dolfinx
print("dolfinx version:", dolfinx.__version__)

if _on_colab:
    from google.colab import files
# --- end canonical: bootstrap ---

In [ ]:
import os, math
from mpi4py import MPI
import ufl
from petsc4py import PETSc
from dolfinx.mesh import create_unit_square, CellType
from dolfinx.fem import functionspace, Function, Constant, form, assemble_scalar
from dolfinx.fem.petsc import NonlinearProblem
import basix.ufl as bufl


In [ ]:
# DD2365_FAST: run N=8,16 only for CI
N_list = [8, 16] if os.environ.get('DD2365_FAST') == '1' else [8, 16, 32, 64, 128]
C_penalty = 1e3
C_nitsche_list = [20, 100, 1000]


## Manufactured solution

Stream function ψ = sin²(πx)sin²(πy) + x²y

Velocity (divergence-free by construction from ψ):
  u_x = π sin²(πx) sin(2πy) + x²
  u_y = −π sin(2πx) sin²(πy) − 2xy

Pressure (mean zero over [0,1]²):
  p = cos(πx) sin(πy)   — ∫p dx = 0 since ∫₀¹cos(πx)dx = 0

Body force f = −Δu + ∇p (computed analytically from the above):
  Δu_x = 2π³ sin(2πy)(2cos(2πx)−1) + 2
  Δu_y = 2π³ sin(2πx)(4sin²(πy)−1)

All of u_ex, p_ex, f enter the forms as UFL SpatialCoordinate expressions.

## Convergence study

**Penalty form** (non-symmetric):

    F((u,p);(v,q)) = ∫ν·∇u:∇v dx − ∫p·div(v) dx + ∫div(u)·q dx
                   + γ∫(u−g)·v ds − ∫f·v dx = 0

**Symmetric Nitsche form** (traction σ(u,p)·n = ∇u·n − pn, ν=1):

    F((u,p);(v,q)) = ∫ν·∇u:∇v dx − ∫p·div(v) dx + ∫div(u)·q dx
                   − ⟨σ(u,p)·n, v⟩_∂Ω               (consistency)
                   − ⟨(∇v·n + qn), u−g⟩_∂Ω          (symmetry)
                   + γ⟨u−g, v⟩_∂Ω                   (penalty)
                   − ∫f·v dx = 0

The adjoint traction is ∇v·n + qn (note +qn, not −qn): IBP of the interior
+∫div(u)·q dx term produces a +⟨u·n, q⟩ boundary flux, whose (v,q)-sided
partner is +qn·(u−g).  Coercivity requires C > C_inv for P2; C = 20 is the
chosen value (verified in this MMS study).

Solved with NonlinearProblem + Newton (1 step — linear).

Error columns: ‖u‖_L2 = ‖u_h−u_ex‖_L2(Ω), ‖u‖_H1 = ‖∇(u_h−u_ex)‖_F,L2(Ω),
‖p‖_L2 = ‖(p_h−mean_h) − (p_ex−mean_ex)‖_L2(Ω),
e_bdy = ‖u_h−g‖_L2(∂Ω).  Quadrature degree 7 (= 2·2+3).

In [ ]:
def run_stokes(N_list, C, method='penalty'):
    """method: 'penalty' or 'nitsche'"""
    pi    = math.pi
    qdeg  = 7  # 2*k+3, k=2 for P2 velocity
    dx_ex = ufl.dx(metadata={'quadrature_degree': qdeg})
    ds_ex = ufl.ds(metadata={'quadrature_degree': qdeg})
    results = []
    for N in N_list:
        msh = create_unit_square(MPI.COMM_WORLD, N, N, cell_type=CellType.triangle)

        P2v = bufl.element('Lagrange', msh.basix_cell(), 2, shape=(2,))
        P1s = bufl.element('Lagrange', msh.basix_cell(), 1)
        TH  = bufl.mixed_element([P2v, P1s])
        W   = functionspace(msh, TH)
        w   = Function(W)
        u, p = ufl.split(w)
        v, q = ufl.split(ufl.TestFunction(W))

        nu    = Constant(msh, PETSc.ScalarType(1.0))
        h_c   = ufl.CellDiameter(msh)
        gamma = Constant(msh, PETSc.ScalarType(C)) / h_c

        x = ufl.SpatialCoordinate(msh)
        ux_ex = pi*ufl.sin(pi*x[0])**2 * ufl.sin(2*pi*x[1]) + x[0]**2
        uy_ex = -pi*ufl.sin(2*pi*x[0]) * ufl.sin(pi*x[1])**2 - 2*x[0]*x[1]
        u_ex_ufl = ufl.as_vector([ux_ex, uy_ex])
        p_ex_ufl = ufl.cos(pi*x[0]) * ufl.sin(pi*x[1])
        lap_ux = 2*pi**3 * ufl.sin(2*pi*x[1]) * (2*ufl.cos(2*pi*x[0]) - 1) + 2.0
        lap_uy = 2*pi**3 * ufl.sin(2*pi*x[0]) * (4*ufl.sin(pi*x[1])**2 - 1)
        dp_dx  = -pi * ufl.sin(pi*x[0]) * ufl.sin(pi*x[1])
        dp_dy  =  pi * ufl.cos(pi*x[0]) * ufl.cos(pi*x[1])
        f_ufl  = ufl.as_vector([-lap_ux + dp_dx, -lap_uy + dp_dy])

        F_vol = (
            nu * ufl.inner(ufl.grad(u), ufl.grad(v)) * ufl.dx
            - p * ufl.div(v) * ufl.dx
            + ufl.div(u) * q * ufl.dx
            - ufl.inner(f_ufl, v) * ufl.dx
        )

        if method == 'nitsche':
            n_vec = ufl.FacetNormal(msh)
            def _bc(g):
                t_u = ufl.dot(ufl.grad(u), n_vec) - p * n_vec
                t_v = ufl.dot(ufl.grad(v), n_vec) + q * n_vec  # +qn: adjoint of +div(u)*q convention
                return (
                    - ufl.inner(t_u, v) * ufl.ds
                    - ufl.inner(t_v, u - g) * ufl.ds
                    + gamma * ufl.inner(u - g, v) * ufl.ds
                )
            F = F_vol + _bc(u_ex_ufl)
        else:
            F = F_vol + gamma * ufl.inner(u - u_ex_ufl, v) * ufl.ds

        problem = NonlinearProblem(
            F, w,
            petsc_options_prefix='mms_stokes_',
            petsc_options={
                'snes_type': 'newtonls',
                'snes_rtol': 1e-12,
                'snes_atol': 1e-14,
                'ksp_type': 'preonly',
                'pc_type': 'lu',
                'pc_factor_mat_solver_type': 'mumps',
            },
        )
        problem.solve()
        n_iters = problem.solver.getIterationNumber()

        one  = Constant(msh, PETSc.ScalarType(1.0))
        vol  = float(msh.comm.allreduce(
            assemble_scalar(form(one * ufl.dx)), op=MPI.SUM))
        p_mean_h  = float(msh.comm.allreduce(
            assemble_scalar(form(p * dx_ex)), op=MPI.SUM)) / vol
        p_ex_mean = float(msh.comm.allreduce(
            assemble_scalar(form(p_ex_ufl * dx_ex)), op=MPI.SUM)) / vol

        eu_ufl = u - u_ex_ufl
        ep_ufl = (p - Constant(msh, PETSc.ScalarType(p_mean_h))) \
               - (p_ex_ufl - p_ex_mean)

        e_L2_u = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(ufl.inner(eu_ufl, eu_ufl) * dx_ex)), op=MPI.SUM))))
        e_H1_u = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(
                ufl.inner(ufl.grad(eu_ufl), ufl.grad(eu_ufl)) * dx_ex)), op=MPI.SUM))))
        e_L2_p = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(ep_ufl**2 * dx_ex)), op=MPI.SUM))))
        e_bdy  = math.sqrt(abs(float(msh.comm.allreduce(
            assemble_scalar(form(ufl.inner(eu_ufl, eu_ufl) * ds_ex)), op=MPI.SUM))))

        h_val = 1.0 / N
        results.append({'N': N, 'h': h_val,
                         'e_L2_u': e_L2_u, 'e_H1_u': e_H1_u,
                         'e_L2_p': e_L2_p, 'e_bdy': e_bdy,
                         'iters': n_iters})
        print(f'N={N:4d}  h={h_val:.4f}  ‖u‖_L2={e_L2_u:.4e}  '
              f'‖u‖_H1={e_H1_u:.4e}  ‖p‖_L2={e_L2_p:.4e}  '
              f'e_bdy={e_bdy:.4e}  iters={n_iters}')
    return results


In [ ]:
print(f'C = {C_penalty:.0e}')
results_default = run_stokes(N_list, C_penalty)


### Results table

In [ ]:
def print_stokes_table(results, label=''):
    if label:
        print(f'--- {label} ---')
    hdr = ('N'.rjust(6) + '  ' + 'h'.rjust(7) + '  ' +
           '‖u‖_L2'.rjust(12) + '  ' + 'r_L2u'.rjust(6) + '  ' +
           '‖u‖_H1'.rjust(12) + '  ' + 'r_H1u'.rjust(6) + '  ' +
           '‖p‖_L2'.rjust(12) + '  ' + 'r_L2p'.rjust(6) + '  ' +
           'e_bdy'.rjust(12)  + '  ' + 'r_bdy'.rjust(6))
    print(hdr)
    for i, r in enumerate(results):
        if i == 0:
            rl2u = rh1u = rl2p = rb = '    —'
        else:
            prev = results[i-1]
            rl2u = f'{math.log2(prev["e_L2_u"]/r["e_L2_u"]):6.2f}'
            rh1u = f'{math.log2(prev["e_H1_u"]/r["e_H1_u"]):6.2f}'
            rl2p = f'{math.log2(prev["e_L2_p"]/r["e_L2_p"]):6.2f}'
            rb   = f'{math.log2(prev["e_bdy" ]/r["e_bdy" ]):6.2f}'
        print(f"{r['N']:>6}  {r['h']:>7.4f}  {r['e_L2_u']:>12.4e}  {rl2u}  "
              f"{r['e_H1_u']:>12.4e}  {rh1u}  {r['e_L2_p']:>12.4e}  {rl2p}  "
              f"{r['e_bdy']:>12.4e}  {rb}")

print_stokes_table(results_default, f'C = {C_penalty:.0e}')


### Penalty sensitivity

In [ ]:
print('\n=== C = 1e1 ===')
results_c1 = run_stokes(N_list, 1e1)
print('\n=== C = 1e5 ===')
results_c5 = run_stokes(N_list, 1e5)


In [ ]:
print_stokes_table(results_c1, 'C = 1e1')
print()
print_stokes_table(results_c5, 'C = 1e5')


## Nitsche method

Symmetric Nitsche BC: primal traction σ(u,p)·n = ∇u·n − pn; adjoint ∇v·n + qn (ν=1).
Three terms: consistency + symmetry + penalty.  C = 20 (P2, chosen value, verified here).
Expected: optimal rates (‖u‖_L2→3, ‖u‖_H1→2, ‖p‖_L2→2) for all C ≥ 20.
e_bdy should scale as h^(k+1) = h³ (velocity) — much better than penalty at same C.

In [ ]:
nitsche_results = {}
for C_n in C_nitsche_list:
    print(f'\n=== Nitsche C = {C_n} ===')
    nitsche_results[C_n] = run_stokes(N_list, C_n, method='nitsche')


In [ ]:
for C_n in C_nitsche_list:
    print()
    print_stokes_table(nitsche_results[C_n], f'Nitsche C = {C_n}')
